# nian41 뼈대(포즈) 추출 — Colab 무료 GPU

순서대로 실행하거나, 메뉴 **런타임 → 모두 실행**을 누른다.

- 시작 전: **런타임 → 런타임 유형 변경 → T4 GPU** 선택
- 2번 칸에서 Google Drive 연결을 허용한다.
- 끊겨도 다시 '모두 실행'하면 끝난 영상은 건너뛰고 이어서 한다.
- 결과: 내 드라이브 `AI-sfa/02_분석자료/포즈/nian41/` 와 `nian41_pose.zip`

In [ ]:
# 1. 준비(1~2분)
!pip -q install ultralytics
import torch; print('GPU:', torch.cuda.is_available() and torch.cuda.get_device_name(0))

In [ ]:
# 2. 내 드라이브 연결(허용 창이 뜨면 계정 선택 → 허용)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. 설정
import os, json, gzip, time, shutil
SRC = '/content/drive/MyDrive/AI-sfa/01_원본영상/_Gemini용_저용량/nian41'
OUT = '/content/drive/MyDrive/AI-sfa/02_분석자료/포즈/nian41'
os.makedirs(OUT, exist_ok=True)
IDS = ["287139", "280835", "280834", "280833", "280832", "280831", "280830", "280598", "280597", "280556", "280544", "280536", "280535", "280534", "280530", "280528", "280527", "280526", "280525", "280523", "280521", "216089", "216085", "216084", "216081", "216079", "216077", "216076", "216075", "216073", "2029588466376511962", "1943223713455051062", "1937399147772137658", "1913589758644908348", "1913481070839742689", "1913231262086463573", "1613887377965211649", "1613887197383622656", "1569962812805644289", "1567691748968906752", "1534078885696000000", "1534075160545878017", "301158", "301142", "287268", "287264", "287263", "287140"]
have = [i for i in IDS if os.path.exists(f'{SRC}/{i}.mp4')]
print('대상', len(IDS), '드라이브에 있음', len(have), '이미 끝남', sum(os.path.exists(f'{OUT}/{i}.jsonl.gz') for i in IDS))

In [ ]:
# 4. 포즈 추출(모든 프레임, 약 10~15분). 영상마다 한 줄씩 진행 상황이 찍힌다.
from ultralytics import YOLO
model = YOLO('yolo11m-pose.pt')
t0 = time.time()
for k, vid in enumerate(have):
    dst = f'{OUT}/{vid}.jsonl.gz'
    if os.path.exists(dst):
        continue
    tmp = f'/content/{vid}.jsonl.gz'
    n = det = 0
    with gzip.open(tmp, 'wt', encoding='utf-8') as f:
        for i, res in enumerate(model.predict(f'{SRC}/{vid}.mp4', stream=True, verbose=False, half=True, imgsz=640, conf=0.25)):
            people = []
            if res.keypoints is not None and len(res.boxes):
                kp = res.keypoints.data.cpu().numpy().round(1).tolist()
                bx = res.boxes.xyxy.cpu().numpy().round(1).tolist()
                cf = res.boxes.conf.cpu().numpy().round(3).tolist()
                people = [{'box': bx[j], 'conf': cf[j], 'kpts': kp[j]} for j in range(len(bx))]
            f.write(json.dumps({'f': i, 'p': people}) + '\n')
            n += 1; det += bool(people)
    shutil.move(tmp, dst)  # 로컬에 다 쓴 뒤 드라이브로 옮김(중간에 끊겨도 반쪽 파일이 남지 않음)
    print(k + 1, len(have), vid, '프레임', n, '사람 검출', det, f'{time.time()-t0:.0f}초', flush=True)
print('끝')

In [ ]:
# 5. 결과를 zip 하나로 묶기(이 파일을 Claude 채팅에 첨부)
import shutil
zp = shutil.make_archive('/content/drive/MyDrive/AI-sfa/02_분석자료/포즈/nian41_pose', 'zip', OUT)
print(zp, round(os.path.getsize(zp)/1e6, 1), 'MB', '파일 수', len([x for x in os.listdir(OUT) if x.endswith('.gz')]))